# Factorial (Java)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/Factorial_java.ipynb)

Chapter 14 — Recursion. Runnable notebook of `Factorial`: the algorithm in Java, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/Factorial_python.ipynb) and [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/Factorial_cpp.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion_java.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. The first cell checks that the JDK is there and installs it if it is missing. Each Java program is written to a file by a `%%writefile` cell, then compiled with `javac` and run with `java` in the next cell.

**Names.** There are no arrays in this chapter: variables and parameters have the same names as in the pseudo-code. The stack and the counters are `static` fields of the class, and the trace is switched on with `trace = true`.

## Setup

On Colab the JDK is usually already there: the cell below checks it and installs it if it is missing.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Factorial

**Input:** a natural number $n$. **Output:** $n!$, with $0! = 1$. $S(n) = 2n + 2 = \Theta(n)$ steps; $n + 1$ calls, depth $n + 1$ (the tree is a path).

Pseudo-code: section 14.4 of the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

In [2]:
%%writefile Factorial.java
public class Factorial
{
static long[] cnt = new long[20];   // cnt[r] = number of executions of line r (numbering of the lecture notes)
static long calls = 0;              // number of calls = nodes of the tree of recursive calls
static int maxDepth = 0;            // depth of the recursion = maximum height of the stack
static long mults = 0;              // multiplications (FastPower)
static long moves = 0;              // moves (Hanoi)
static int ev = 0;                  // counter of the events of the trace
static java.util.ArrayList<Integer> stack = new java.util.ArrayList<>();   // the call stack: the arguments of the active calls
static long[] seen = new long[64];  // seen[x] = number of calls with argument x
static boolean trace = false;       // true: print the trace of the execution

static void reset()
{
    java.util.Arrays.fill(cnt, 0);
    java.util.Arrays.fill(seen, 0);
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0;
    stack.clear();
}

static long total()
{
    long s = 0;
    for (long c : cnt) s += c;
    return s;
}

static void printSteps(int... lines)
{
    System.out.println("Line | Executions");
    for (int r : lines) System.out.println(String.format("%4d | %10d", r, cnt[r]));
    System.out.println("Total steps S = " + total());
}

static void push(int x)             // a call starts: its frame goes on top of the stack
{
    calls++;
    stack.add(x);
    if (stack.size() > maxDepth) maxDepth = stack.size();
    if (x >= 0 && x < 64) seen[x]++;
}

static void pop()                   // the call returns: its frame leaves the stack
{
    stack.remove(stack.size() - 1);
}

static void event(String text)      // numbered line of the trace, with the stack
{
    ev++;
    System.out.println(String.format("%2d. %s   stack: %s", ev, text, stack));
}

static void say(String call, String action)   // line of the trace: the call, indented one level for every active call
{
    System.out.println(String.format("%-28s| %s", "  ".repeat(stack.size() - 1) + call, action));
}

    // Factorial(n) correct only up to n = 20 (in C++ and Java 64 bits overflow beyond)
    static long factorial(int n)
    {
        push(n); cnt[2]++;
        if (n == 0)
        {
            cnt[3]++;
            if (trace) event("Factorial(0): n = 0 => base case");
            pop();
            if (trace) event("Factorial(0) returns 1");
            return 1;
        }
        if (trace) event(String.format("Factorial(%d): %d != 0 => must compute %d * Factorial(%d) and waits", n, n, n, n - 1));
        cnt[5]++;
        long f = factorial(n - 1);
        long r = n * f;
        pop();
        if (trace) event(String.format("Factorial(%d) returns %d * %d = %d", n, n, f, r));
        return r;
    }

    public static void main(String[] args)
    {
        // Example of the lecture notes: Factorial(4), with the stack
        int n = 4;
        reset();
        trace = true;
        long r = factorial(n);
        trace = false;
        System.out.printf("Factorial(%d) = %d\n", n, r);
        System.out.printf("Calls = %d (nodes of the tree), depth of the recursion = %d\n", calls, maxDepth);
        printSteps(2, 3, 5);
        System.out.printf("S(n) = 2n + 2 = %d\n", 2 * n + 2);

        // Calls, depth and steps as n varies
        System.out.printf("\n");
        System.out.printf("  n |                   n! |    calls |       depth | S(n) | 2n + 2\n");
        int[] values = {0, 1, 2, 3, 4, 5, 10, 15, 20};
        for (int v : values)
        {
            reset();
            long f = factorial(v);
            System.out.printf("%3d | %20d | %8d | %11d | %4d | %d\n", v, f, calls, maxDepth, total(), 2 * v + 2);
        }
    }
}

Writing Factorial.java


In [3]:
!javac Factorial.java && java Factorial

 1. Factorial(4): 4 != 0 => must compute 4 * Factorial(3) and waits   stack: [4]
 2. Factorial(3): 3 != 0 => must compute 3 * Factorial(2) and waits   stack: [4, 3]
 3. Factorial(2): 2 != 0 => must compute 2 * Factorial(1) and waits   stack: [4, 3, 2]
 4. Factorial(1): 1 != 0 => must compute 1 * Factorial(0) and waits   stack: [4, 3, 2, 1]
 5. Factorial(0): n = 0 => base case   stack: [4, 3, 2, 1, 0]
 6. Factorial(0) returns 1   stack: [4, 3, 2, 1]
 7. Factorial(1) returns 1 * 1 = 1   stack: [4, 3, 2]
 8. Factorial(2) returns 2 * 1 = 2   stack: [4, 3]
 9. Factorial(3) returns 3 * 2 = 6   stack: [4]
10. Factorial(4) returns 4 * 6 = 24   stack: []
Factorial(4) = 24
Calls = 5 (nodes of the tree), depth of the recursion = 5
Line | Executions
   2 |          5
   3 |          1
   5 |          4
Total steps S = 10
S(n) = 2n + 2 = 10

  n |                   n! |    calls |       depth | S(n) | 2n + 2
  0 |                    1 |        1 |           1 |    2 | 2
  1 |                    1 |

**Try it yourself.** Change $n$ and watch the stack grow and then empty.

In [4]:
%%writefile ProvaFactorial.java
public class ProvaFactorial
{
    public static void main(String[] args)
    {
        int n = 6;        // <-- change n (20 at most)
        Factorial.reset();
        Factorial.trace = true;
        long r = Factorial.factorial(n);
        Factorial.trace = false;
        System.out.printf("Factorial(%d) = %d, calls = %d, depth = %d, S = %d\n", n, r, Factorial.calls, Factorial.maxDepth, Factorial.total());
    }
}

Writing ProvaFactorial.java


In [5]:
!javac ProvaFactorial.java && java ProvaFactorial

 1. Factorial(6): 6 != 0 => must compute 6 * Factorial(5) and waits   stack: [6]
 2. Factorial(5): 5 != 0 => must compute 5 * Factorial(4) and waits   stack: [6, 5]
 3. Factorial(4): 4 != 0 => must compute 4 * Factorial(3) and waits   stack: [6, 5, 4]
 4. Factorial(3): 3 != 0 => must compute 3 * Factorial(2) and waits   stack: [6, 5, 4, 3]
 5. Factorial(2): 2 != 0 => must compute 2 * Factorial(1) and waits   stack: [6, 5, 4, 3, 2]
 6. Factorial(1): 1 != 0 => must compute 1 * Factorial(0) and waits   stack: [6, 5, 4, 3, 2, 1]
 7. Factorial(0): n = 0 => base case   stack: [6, 5, 4, 3, 2, 1, 0]
 8. Factorial(0) returns 1   stack: [6, 5, 4, 3, 2, 1]
 9. Factorial(1) returns 1 * 1 = 1   stack: [6, 5, 4, 3, 2]
10. Factorial(2) returns 2 * 1 = 2   stack: [6, 5, 4, 3]
11. Factorial(3) returns 3 * 2 = 6   stack: [6, 5, 4]
12. Factorial(4) returns 4 * 6 = 24   stack: [6, 5]
13. Factorial(5) returns 5 * 24 = 120   stack: [6]
14. Factorial(6) returns 6 * 120 = 720   stack: []
Factorial(6) = 720, c

---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).